# Crime Hotspot Classification

## PART 1 — Training Pipeline (2021–2023)

> ⚠️ The 2024 dataset is **not loaded anywhere in this section**.
> Models are trained, then serialised to disk.
> Part 2 (bottom of notebook) loads 2024 as a fully fresh holdout.

In [ ]:
import pandas as pd
import numpy as np
import re
import joblib

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, accuracy_score, roc_curve
)

import matplotlib.pyplot as plt
import seaborn as sns

print("Imports complete.")


### Step 1 · Load Training Data

**Only links 1–3 (2021–2023) are loaded here.** Link 4 does not exist yet.

In [ ]:
def to_csv_url(link):
    m = re.search(r"/d/([A-Za-z0-9_-]+)", link)
    if not m:
        raise ValueError(f"Not a valid Drive file URL: {link}")
    return f"https://drive.google.com/uc?export=download&id={m.group(1)}"

TRAIN_LINKS = [
    "https://drive.google.com/file/d/1rgFQUYCOhxjyfcCK90oQCrYn7uVEn8Fn/view?usp=sharing",  # 2021
    "https://drive.google.com/file/d/1aJ_cmeHrXQzutYtAxb-wGqJV75z8jv8u/view?usp=sharing",  # 2022
    "https://drive.google.com/file/d/1C1iAJmsXjx-Ux-xC77EQ3-cOUO5Kkiru/view?usp=sharing",  # 2023
]

train_raw_df = pd.concat(
    [pd.read_csv(to_csv_url(link)) for link in TRAIN_LINKS],
    ignore_index=True
)

print(f"Training raw shape (2021-2023): {train_raw_df.shape}")
print(f"Columns: {train_raw_df.columns.tolist()}")


### Step 2 · Clean Training Data

In [ ]:
COLUMNS_TO_DROP = [
    'X', 'Y', 'CCN', 'REPORT_DAT', 'BLOCK',
    'XBLOCK', 'YBLOCK', 'BLOCK_GROUP', 'CENSUS_TRACT',
    'VOTING_PRECINCT', 'BID', 'END_DATE', 'OBJECTID', 'OCTO_RECORD_ID'
]

CRITICAL_COLS = [
    'START_DATE', 'LATITUDE', 'LONGITUDE',
    'DISTRICT', 'PSA', 'NEIGHBORHOOD_CLUSTER'
]

def clean_df(df, label, date_start, date_end):
    df = df.copy()
    drop_cols = [c for c in COLUMNS_TO_DROP if c in df.columns]
    df.drop(columns=drop_cols, inplace=True)
    print(f"[{label}] After column drop: {df.shape}")

    before = len(df)
    df.dropna(subset=[c for c in CRITICAL_COLS if c in df.columns], inplace=True)
    print(f"[{label}] Dropped {before - len(df)} rows with missing critical cols")

    df['START_DATE'] = pd.to_datetime(df['START_DATE'], errors='coerce', utc=True)
    df.dropna(subset=['START_DATE'], inplace=True)

    start_ts = pd.Timestamp(date_start, tz='UTC')
    end_ts   = pd.Timestamp(date_end,   tz='UTC')
    before = len(df)
    df = df[(df['START_DATE'] >= start_ts) & (df['START_DATE'] <= end_ts)]
    print(f"[{label}] Dropped {before - len(df)} rows outside date range")

    if 'WARD' in df.columns:
        before = len(df)
        df.dropna(subset=['WARD'], inplace=True)
        print(f"[{label}] Dropped {before - len(df)} rows with missing WARD")

    print(f"[{label}] Final clean shape: {df.shape}\n")
    return df.reset_index(drop=True)

train_df = clean_df(train_raw_df, 'TRAIN', '2021-01-01', '2023-12-31 23:59:59')


### Step 3 · Time Features

In [ ]:
def add_time_features(df):
    df = df.copy()
    df['hour']         = df['START_DATE'].dt.hour
    df['day_of_week']  = df['START_DATE'].dt.dayofweek
    df['day_of_year']  = df['START_DATE'].dt.dayofyear
    df['month']        = df['START_DATE'].dt.month
    df['year']         = df['START_DATE'].dt.year
    df['quarter']      = df['START_DATE'].dt.quarter
    df['week_of_year'] = df['START_DATE'].dt.isocalendar().week.astype(int)
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
    return df

train_df = add_time_features(train_df)
print(f"Time features added. Year range: {train_df['year'].min()} – {train_df['year'].max()}")


### Step 4 · One-Hot Encoding

Fitted on 2021–2023 only. **The OHE column vocabulary is saved to disk** so the test pipeline can apply the exact same encoding to 2024 data.

In [ ]:
OHE_COLS = ['OFFENSE', 'SHIFT', 'METHOD', 'DISTRICT']

train_df['DISTRICT'] = train_df['DISTRICT'].astype(str)

train_dummies  = pd.get_dummies(train_df[OHE_COLS], prefix=OHE_COLS)
TRAIN_OHE_COLS = train_dummies.columns.tolist()   # ← vocabulary fitted on train

train_df = pd.concat([train_df, train_dummies], axis=1).drop(columns=OHE_COLS)

# Save vocabulary so Part 2 can reuse it without touching training data
joblib.dump(TRAIN_OHE_COLS, 'train_ohe_cols.pkl')

print(f"OHE complete. Train shape: {train_df.shape}")
print(f"OHE vocabulary size: {len(TRAIN_OHE_COLS)} columns — saved to train_ohe_cols.pkl")


### Step 5 · Weekly Aggregation per Neighbourhood Cluster

In [ ]:
def build_weekly_agg(df, label, all_clusters=None):
    weekly_counts = (
        df.groupby(['year', 'week_of_year', 'NEIGHBORHOOD_CLUSTER'])
          .size()
          .reset_index(name='crime_count')
    )

    clusters = all_clusters if all_clusters is not None else df['NEIGHBORHOOD_CLUSTER'].unique()
    years    = range(df['year'].min(), df['year'].max() + 1)
    weeks    = range(1, 54)

    full_index = pd.MultiIndex.from_product(
        [years, weeks, clusters],
        names=['year', 'week_of_year', 'NEIGHBORHOOD_CLUSTER']
    )

    weekly_agg = (
        weekly_counts
        .set_index(['year', 'week_of_year', 'NEIGHBORHOOD_CLUSTER'])
        .reindex(full_index, fill_value=0)
        .reset_index()
        .sort_values(['NEIGHBORHOOD_CLUSTER', 'year', 'week_of_year'])
        .reset_index(drop=True)
    )

    print(f"[{label}] Weekly aggregated shape: {weekly_agg.shape}")
    return weekly_agg

# Save cluster set so Part 2 uses the same clusters
TRAIN_CLUSTERS = train_df['NEIGHBORHOOD_CLUSTER'].unique()
joblib.dump(TRAIN_CLUSTERS, 'train_clusters.pkl')
print(f"Cluster set ({len(TRAIN_CLUSTERS)} clusters) saved to train_clusters.pkl")

train_weekly = build_weekly_agg(train_df, 'TRAIN', all_clusters=TRAIN_CLUSTERS)


### Step 6 · Target Variable and Lag / Rolling Features

In [ ]:
HOTSPOT_THRESHOLD = 20

def add_target_and_lags(df, label):
    df = df.copy()

    df['crime_count_next_week'] = (
        df.groupby('NEIGHBORHOOD_CLUSTER')['crime_count'].shift(-1)
    )
    df['is_hotspot_next_week'] = (
        (df['crime_count_next_week'] > HOTSPOT_THRESHOLD)
        .fillna(False).astype(int)
    )

    df['crime_count_lag1'] = df.groupby('NEIGHBORHOOD_CLUSTER')['crime_count'].shift(1)
    df['crime_count_lag4'] = df.groupby('NEIGHBORHOOD_CLUSTER')['crime_count'].shift(4)

    shifted = df.groupby('NEIGHBORHOOD_CLUSTER')['crime_count'].shift(1)
    df['crime_count_roll_mean4'] = shifted.rolling(window=4, min_periods=1).mean()
    df['crime_count_roll_std4']  = shifted.rolling(window=4, min_periods=1).std()

    print(f"[{label}] Shape after target + lags: {df.shape}")
    return df

train_weekly = add_target_and_lags(train_weekly, 'TRAIN')


### Step 7 · Feature Selection & NaN Cleanup

In [ ]:
FEATURE_COLS = [
    'crime_count',
    'crime_count_lag1',
    'crime_count_lag4',
    'crime_count_roll_mean4',
    'crime_count_roll_std4',
    'year',
    'week_of_year',
]
TARGET_COL = 'is_hotspot_next_week'

# Save feature list so Part 2 selects identical columns
joblib.dump(FEATURE_COLS, 'feature_cols.pkl')

def select_and_clean(df, label):
    X = df[FEATURE_COLS].copy()
    y = df[TARGET_COL].copy()
    nan_mask = X.notna().all(axis=1)
    dropped  = (~nan_mask).sum()
    X, y = X[nan_mask], y[nan_mask]
    print(f"[{label}] Dropped {dropped} NaN rows")
    print(f"[{label}] Final → X: {X.shape}, y: {y.shape}")
    print(f"[{label}] Hotspot rate: {y.mean():.3f}\n")
    return X, y

X_train, y_train = select_and_clean(train_weekly, 'TRAIN')


### Step 8 · Train Random Forest

Trained on 2021–2023. Model serialised to disk — **no test data seen yet**.

In [ ]:
print("--- Training Random Forest (2021-2023) ---")

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

joblib.dump(rf_model, 'rf_model.pkl')
print("Random Forest trained and saved to rf_model.pkl")

# Training-set self-check (not a test — just sanity)
y_pred_train_rf = rf_model.predict(X_train)
print(f"Train accuracy (sanity): {accuracy_score(y_train, y_pred_train_rf):.4f}")
print("(High train accuracy is expected — this is NOT the evaluation metric)")


### Step 9 · Train XGBoost

`scale_pos_weight` derived from 2021–2023 class distribution only.

In [ ]:
print("--- Training XGBoost (2021-2023) ---")

count_neg = (y_train == 0).sum()
count_pos = (y_train == 1).sum()
scale_pos_weight_val = count_neg / count_pos if count_pos > 0 else 1
print(f"scale_pos_weight (from train): {scale_pos_weight_val:.2f}")

xgb_model = XGBClassifier(
    objective='binary:logistic',
    scale_pos_weight=scale_pos_weight_val,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)

joblib.dump(xgb_model, 'xgb_model.pkl')
print("XGBoost trained and saved to xgb_model.pkl")

y_pred_train_xgb = xgb_model.predict(X_train)
print(f"Train accuracy (sanity): {accuracy_score(y_train, y_pred_train_xgb):.4f}")
print("(High train accuracy is expected — this is NOT the evaluation metric)")


---

# ═══════════════════════════════════════════════
# HOLDOUT TEST — 2024 DATA INTRODUCED HERE
# ═══════════════════════════════════════════════

> The cells above are **complete and frozen**.
> Nothing below existed during training.
> Models and vocabulary are loaded from disk — no retraining, no peeking.

**Pipeline for 2024:**
1. Load raw 2024 CSV (link 4)
2. Apply identical cleaning, time features, OHE (using saved train vocabulary)
3. Weekly aggregation (using saved train clusters)
4. Lag features + NaN drop
5. Load saved models → evaluate

### Test Step 1 · Load 2024 Holdout Data

In [ ]:
TEST_LINK = "https://drive.google.com/file/d/1USTg4Dj60ktjZRMqCNU44JtpfXQ9CWp9/view?usp=sharing"  # 2024

test_raw_df = pd.read_csv(to_csv_url(TEST_LINK))
print(f"2024 raw shape: {test_raw_df.shape}")


### Test Step 2 · Clean 2024 Data

In [ ]:
test_df = clean_df(test_raw_df, 'TEST-2024', '2024-01-01', '2024-12-31 23:59:59')
test_df = add_time_features(test_df)
print(f"2024 after cleaning + time features: {test_df.shape}")
print(f"Year range: {test_df['year'].min()} – {test_df['year'].max()}")


### Test Step 3 · One-Hot Encode 2024 Using **Saved Training Vocabulary**

The test set is reindexed to match `train_ohe_cols.pkl` exactly.
Any category in 2024 not seen in training is silently discarded.

In [ ]:
# Load the exact vocabulary fitted on 2021-2023
TRAIN_OHE_COLS = joblib.load('train_ohe_cols.pkl')

test_df['DISTRICT'] = test_df['DISTRICT'].astype(str)

test_dummies = pd.get_dummies(test_df[OHE_COLS], prefix=OHE_COLS)
test_dummies = test_dummies.reindex(columns=TRAIN_OHE_COLS, fill_value=0)

test_df = pd.concat([test_df, test_dummies], axis=1).drop(columns=OHE_COLS)

print(f"2024 after OHE: {test_df.shape}")
print("OHE vocabulary matches training exactly — no leakage.")


### Test Step 4 · Weekly Aggregation (using saved cluster set)

In [ ]:
TRAIN_CLUSTERS = joblib.load('train_clusters.pkl')

test_weekly = build_weekly_agg(test_df, 'TEST-2024', all_clusters=TRAIN_CLUSTERS)
test_weekly = add_target_and_lags(test_weekly, 'TEST-2024')

print("\nFirst few rows of test_weekly:")
print(test_weekly.head())


### Test Step 5 · Feature Selection & NaN Drop

Early-2024 rows with NaN lag features are dropped (no December 2023 context exists in the 2024 file).

In [ ]:
FEATURE_COLS = joblib.load('feature_cols.pkl')

X_test, y_test = select_and_clean(test_weekly, 'TEST-2024')

print(f"Class distribution in 2024 test set:")
print(y_test.value_counts(normalize=True).round(3))


### Test Step 6 · Load Saved Models and Evaluate on 2024

In [ ]:
rf_model  = joblib.load('rf_model.pkl')
xgb_model = joblib.load('xgb_model.pkl')
print("Models loaded from disk.")

# Random Forest
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

# XGBoost
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("\n" + "="*50)
print("RANDOM FOREST — 2024 Holdout Results")
print("="*50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test, y_prob_rf):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Not Hotspot', 'Hotspot']))

print("\n" + "="*50)
print("XGBOOST — 2024 Holdout Results")
print("="*50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test, y_prob_xgb):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=['Not Hotspot', 'Hotspot']))


### Test Step 7 · Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('2024 Holdout Evaluation — Models Trained on 2021-2023 Only', fontsize=13, fontweight='bold')

# RF Confusion Matrix
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred: Not Hotspot', 'Pred: Hotspot'],
            yticklabels=['Act: Not Hotspot',  'Act: Hotspot'])
axes[0].set_title('Random Forest — Confusion Matrix')

# XGBoost Confusion Matrix
sns.heatmap(confusion_matrix(y_test, y_pred_xgb), annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Pred: Not Hotspot', 'Pred: Hotspot'],
            yticklabels=['Act: Not Hotspot',  'Act: Hotspot'])
axes[1].set_title('XGBoost — Confusion Matrix')

# ROC Comparison
fpr_rf,  tpr_rf,  _ = roc_curve(y_test, y_prob_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)
auc_rf  = roc_auc_score(y_test, y_prob_rf)
auc_xgb = roc_auc_score(y_test, y_prob_xgb)

axes[2].plot(fpr_rf,  tpr_rf,  color='blue',  linestyle='--', label=f'RF      (AUC = {auc_rf:.4f})')
axes[2].plot(fpr_xgb, tpr_xgb, color='green', label=f'XGBoost (AUC = {auc_xgb:.4f})')
axes[2].plot([0, 1], [0, 1], 'k--')
axes[2].set_xlabel('False Positive Rate')
axes[2].set_ylabel('True Positive Rate')
axes[2].set_title('ROC Comparison — 2024 Holdout')
axes[2].legend()

plt.tight_layout()
plt.show()
